In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet(
    "../data/processed/feature_dataset.parquet"
)

print(df.shape)

(1737509, 238)


In [3]:
# chronological split

n = len(df)

train_end = int(n * 0.70)

val_end = int(n * 0.85)

print(train_end)
print(val_end)

1216256
1476882


In [4]:
train_df = df.iloc[:train_end]

val_df = df.iloc[
    train_end:val_end
]

test_df = df.iloc[
    val_end:
]

In [5]:
print(
    "Train:",
    train_df.shape
)

print(
    "Val:",
    val_df.shape
)

print(
    "Test:",
    test_df.shape
)

Train: (1216256, 238)
Val: (260626, 238)
Test: (260627, 238)


In [6]:
for name, split in [

    ("Train", train_df),

    ("Val", val_df),

    ("Test", test_df)

]:

    print("\n", name)

    print(
        split["TimeStamp"].min()
    )

    print(
        split["TimeStamp"].max()
    )


 Train
2022-01-03 23:46:00
2024-05-30 00:04:00

 Val
2024-05-30 00:05:00
2024-11-30 21:09:00

 Test
2024-11-30 21:10:00
2025-06-23 20:29:00


In [8]:
train_df.to_parquet(
    "../data/processed/train.parquet",
    index=False
)

val_df.to_parquet(
    "../data/processed/val.parquet",
    index=False
)

test_df.to_parquet(
    "../data/processed/test.parquet",
    index=False
)

In [10]:
TARGET = "03TIC_1009.PV"

for split_name, split_df in [
    ("Train", train_df),
    ("Val", val_df),
    ("Test", test_df)
]:

    alarms = (
        split_df[TARGET] < 30.5
    ).sum()

    print(
        split_name,
        alarms
    )

Train 172
Val 54
Test 31


In [11]:
for split_name, split_df in [
    ("Train", train_df),
    ("Val", val_df),
    ("Test", test_df)
]:

    pct = (
        (split_df[TARGET] < 30.5)
        .mean()
        * 100
    )

    print(
        split_name,
        f"{pct:.4f}%"
    )

Train 0.0141%
Val 0.0207%
Test 0.0119%


In [ ]:
## Chronological Data Splitting

Industrial forecasting systems operate on future unseen data. Therefore, preserving temporal ordering during model development is critical to avoid information leakage.

A chronological split strategy was adopted instead of random sampling or cross-validation. The dataset was partitioned according to time order as follows:

* Training Set: earliest 70% of observations
* Validation Set: next 15% of observations
* Test Set: latest 15% of observations

### Split Statistics

| Split      |      Rows |
| ---------- | --------: |
| Train      | 1,216,256 |
| Validation |   260,626 |
| Test       |   260,627 |

### Temporal Coverage

| Split      | Time Range                |
| ---------- | ------------------------- |
| Train      | January 2022 – May 2024   |
| Validation | May 2024 – November 2024  |
| Test       | November 2024 – June 2025 |

This chronological partitioning ensures that all validation and testing observations occur strictly after the training period, thereby mimicking a real-world deployment scenario.

### Alarm Distribution Verification

The forecasting threshold of 30.5 was used to verify the presence of alarm samples within each split.

| Split      | Alarm Samples |
| ---------- | ------------: |
| Train      |           172 |
| Validation |            54 |
| Test       |            31 |

Alarm sample proportions remained within a comparable range across all three splits, indicating that each dataset contains meaningful alarm behavior for evaluation purposes.

### Conclusion

The resulting train, validation, and test partitions preserve temporal ordering, prevent data leakage, and provide sufficient alarm representation for forecasting model evaluation.
